In [ ]:
"""Regenerate figures/fig_methods_pipeline.png from provenance/pipeline_metrics.json.

Run from the repository root (or from figures/notebooks/ -- the path probe below
handles both). Requires only matplotlib + the project-local shared_style module.
"""
import json
import os
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle

HERE = os.path.dirname(os.path.abspath("__file__" if "__file__" not in dir() else __file__))
ROOT = os.getcwd()
while not os.path.exists(os.path.join(ROOT, "provenance", "pipeline_metrics.json")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        raise SystemExit("provenance/pipeline_metrics.json not found above cwd")
    ROOT = parent
sys.path.insert(0, os.path.join(ROOT, "figures", "notebooks"))

from shared_style import COLORS, apply_style

apply_style()

with open(os.path.join(ROOT, "provenance", "pipeline_metrics.json")) as fh:
    M = json.load(fh)

OUT = os.path.join(ROOT, "figures", "fig_methods_pipeline.png")
print(M["review_title"], "|", len(M["phases"]), "ledger entries")


In [ ]:
# Owner palette and the per-phase box geometry.
OWNER_COLOR = {
    "litreview": COLORS["primary_green"],
    "dataml": COLORS["primary_blue"],
    "critic": COLORS["primary_purple"],
}
OWNER_LABEL = {
    "litreview": "LITREVIEW (scientific judgment)",
    "dataml": "DATAML (mechanical)",
    "critic": "Blinded critic / auditor",
}

phases = M["phases"]
N_COL = 11
rowsize = [phases[:N_COL], phases[N_COL:]]
print([len(r) for r in rowsize])


In [ ]:
fig = plt.figure(figsize=(13.5, 9.6))
gs = fig.add_gridspec(
    3, 3, height_ratios=[1.00, 0.92, 0.92], hspace=0.52, wspace=0.34,
    left=0.055, right=0.985, top=0.935, bottom=0.075,
)

# ---------------------------------------------------------------- panel A
axA = fig.add_subplot(gs[0, :])
axA.set_xlim(0, N_COL)
axA.set_ylim(-0.25, 2.35)
axA.axis("off")

for r, band in enumerate(rowsize):
    y = 1.25 - 1.25 * r
    for c, ph in enumerate(band):
        done = ph["status"] == "complete"
        face = OWNER_COLOR[ph["owner"]]
        box = FancyBboxPatch(
            (c + 0.06, y), 0.88, 0.78,
            boxstyle="round,pad=0.015,rounding_size=0.06",
            linewidth=1.4,
            edgecolor=face if done else COLORS["neutral_gray"],
            facecolor=face if done else "white",
            alpha=1.0 if done else 0.55,
            linestyle="-" if done else (0, (3, 2)),
        )
        axA.add_patch(box)
        axA.text(c + 0.5, y + 0.60, ph["id"], ha="center", va="center",
                 fontsize=11, fontweight="bold",
                 color="white" if done else COLORS["neutral_gray"])
        name = ph["name"]
        words, line, lines = name.split(), "", []
        for w in words:
            trial = (line + " " + w).strip()
            if len(trial) > 11 and line:
                lines.append(line)
                line = w
            else:
                line = trial
        lines.append(line)
        axA.text(c + 0.5, y + 0.38, "\n".join(lines[:3]), ha="center", va="top",
                 fontsize=6.1, linespacing=1.12,
                 color="white" if done else COLORS["neutral_gray"])
        if done:
            axA.text(c + 0.88, y + 0.70, "\u2713", ha="center", va="center",
                     fontsize=8, color="white")

axA.annotate("", xy=(N_COL - 0.02, 1.64), xytext=(0.02, 1.64),
             arrowprops=dict(arrowstyle="-|>", color=COLORS["light_gray"], lw=1.1))
axA.annotate("", xy=(N_COL - 0.02, 0.39), xytext=(0.02, 0.39),
             arrowprops=dict(arrowstyle="-|>", color=COLORS["light_gray"], lw=1.1))

handles = [Rectangle((0, 0), 1, 1, facecolor=OWNER_COLOR[k], edgecolor="none")
           for k in ("litreview", "dataml", "critic")]
handles.append(Rectangle((0, 0), 1, 1, facecolor="white",
                         edgecolor=COLORS["neutral_gray"], linestyle=(0, (3, 2))))
labels = [OWNER_LABEL[k] for k in ("litreview", "dataml", "critic")]
labels.append("runs after deployment")
axA.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, -0.20),
           ncol=4, frameon=False, fontsize=8.2)

pc = M["phase_counts"]
axA.set_title(
    "A  Pipeline architecture \u2014 {} numbered phases plus sub-phase 20a "
    "({} ledger entries); {} complete at the Phase-20a refresh".format(
        pc["numbered_phases"], pc["ledger_entries"], pc["complete"]),
    fontsize=10.5, loc="left", pad=8)




In [ ]:
# ---------------------------------------------------------------- panel B
axB = fig.add_subplot(gs[1, 0])
corp = M["corpus"]
bars = [
    ("Papers reviewed", corp["papers_reviewed_total"], COLORS["neutral_gray"]),
    ("Unique DOIs", corp["unique_dois"], COLORS["primary_blue"]),
    ("Findings extracted", corp["findings"], COLORS["primary_green"]),
    ("  full-text sourced", corp["fulltext_findings"], COLORS["fulltext_marker"]),
    ("  abstract-only", corp["abstract_only_findings"], COLORS["abstract_marker"]),
]
ypos = range(len(bars))[::-1]
axB.barh(list(ypos), [b[1] for b in bars], color=[b[2] for b in bars], height=0.62)
axB.set_yticks(list(ypos))
axB.set_yticklabels([b[0] for b in bars], fontsize=8.4)
for y, b in zip(ypos, bars):
    axB.text(b[1] + 14, y, str(b[1]), va="center", fontsize=8.4)
axB.set_xlim(0, max(b[1] for b in bars) * 1.20)
axB.set_xlabel("count", fontsize=8.6)
axB.set_title("B  Evidence corpus", fontsize=10, loc="left", pad=6)
axB.spines[["top", "right"]].set_visible(False)


In [ ]:
# ---------------------------------------------------------------- panel C
axC = fig.add_subplot(gs[1, 1:])
pcl = M["per_cluster"]
x = range(len(pcl))
ft = [c["fulltext"] for c in pcl]
ab = [c["abstract_only"] for c in pcl]
axC.bar(list(x), ft, color=COLORS["fulltext_marker"], label="full text", width=0.68)
axC.bar(list(x), ab, bottom=ft, color=COLORS["abstract_marker"],
        label="abstract only", width=0.68)
for i, c in enumerate(pcl):
    axC.text(i, c["findings"] + 3, "{:.0f}%".format(c["fulltext_pct"]),
             ha="center", fontsize=7.6, color=COLORS["neutral_gray"])
axC.set_xticks(list(x))
axC.set_xticklabels([c["cluster"].replace("cluster_", "C") for c in pcl],
                    fontsize=8.0)
axC.set_xlabel("evidence cluster (names in Methods \u00a7Search Strategy)", fontsize=8.2)
axC.set_ylabel("findings", fontsize=8.6)
axC.set_ylim(0, max(c["findings"] for c in pcl) * 1.22)
axC.legend(frameon=False, fontsize=8, ncol=2, loc="upper right")
axC.set_title(
    "C  Text access by evidence cluster \u2014 overall full-text share {:.1f}% "
    "({}/{} findings)".format(100 * corp["overall_fulltext_rate"],
                              corp["fulltext_findings"], corp["findings"]),
    fontsize=10, loc="left", pad=6)
axC.spines[["top", "right"]].set_visible(False)



In [ ]:
# ---------------------------------------------------------------- panel D
axD = fig.add_subplot(gs[2, 0])
ex = M["extraction"]
rep = ex["replication"]
seg = [("independently\nreplicated", rep["independently_replicated"], COLORS["primary_green"]),
       ("replication\nunknown", rep["replication_unknown"], COLORS["light_gray"]),
       ("contested", rep["contested"], COLORS["conflict_marker"])]
left = 0
for lab, v, col in seg:
    axD.barh([0], [v], left=left, color=col, height=0.5)
    axD.text(left + v / 2, 0, "{}\n{:.0f}%".format(v, 100 * v / ex["findings"]),
             ha="center", va="center", fontsize=7.8,
             color="white" if col != COLORS["light_gray"] else "black")
    axD.text(left + v / 2, -0.44, lab, ha="center", va="top", fontsize=7.2,
             color=COLORS["neutral_gray"], linespacing=1.2)
    left += v
axD.set_xlim(0, ex["findings"])
axD.set_ylim(-1.0, 0.6)
axD.axis("off")
axD.set_title("D  Replication status of {} findings".format(ex["findings"]),
              fontsize=10, loc="left", pad=6)


In [ ]:
# ---------------------------------------------------------------- panel E
axE = fig.add_subplot(gs[2, 1])
met_bars = [
    ("effect size\nrecorded", ex["with_effect_size_pct"], COLORS["primary_blue"]),
    ("sample size\nrecorded", ex["with_sample_size_pct"], COLORS["primary_orange"]),
    ("schema\ncompliance", 100 * ex["compliance_rate_strict"], COLORS["primary_green"]),
]
axE.bar([0, 1, 2], [m[1] for m in met_bars], color=[m[2] for m in met_bars], width=0.56)
for i, m in enumerate(met_bars):
    axE.text(i, m[1] + 2.2, "{:.1f}%".format(m[1]), ha="center", fontsize=8.4)
axE.set_xticks([0, 1, 2])
axE.set_xticklabels([m[0] for m in met_bars], fontsize=7.4, linespacing=1.25)
axE.set_ylim(0, 112)
axE.set_ylabel("% of findings", fontsize=8.6)
axE.set_title("E  Extraction completeness", fontsize=10, loc="left", pad=6)
axE.spines[["top", "right"]].set_visible(False)


In [ ]:
# ---------------------------------------------------------------- panel F
axF = fig.add_subplot(gs[2, 2])
axF.axis("off")
cit, figm, srch = M["citations"], M["figures"], M["search"]
lines = [
    ("DOIs resolved on CrossRef", "{}/{}".format(cit["crossref_resolved"], cit["dois_mapped"])),
    ("BibTeX entries", str(cit["bib_entries"])),
    ("Citation occurrences, body", str(cit["cite_occurrences_body_sections"])),
    ("Unresolved / orphan keys", "{} / {}".format(cit["unresolved_keys"], cit["orphan_bib_entries"])),
    ("Section figures : notebooks", "{} : {}".format(figm["section_figures"], figm["notebooks"])),
    ("Figure comparisons audited", "{} (caveat {}, pass {})".format(
        figm["figure_audit_comparisons"], figm["audit_CAVEAT"], figm["audit_PASS"])),
    ("Query strings logged", "{} (of {} clusters: {})".format(
        srch["query_strings_logged_total"], 10, srch["clusters_logging_queries"])),
    ("Snowball rounds", str(srch["snowball_rounds"])),
    ("Session frames", "{} over {:.1f} h".format(
        M["frames"]["total"], M["frames"]["elapsed_hours"])),
]
for i, (k, v) in enumerate(lines):
    y = 0.95 - i * 0.108
    axF.text(0.0, y, k, fontsize=7.9, va="center", color=COLORS["neutral_gray"])
    axF.text(1.0, y, v, fontsize=7.9, va="center", ha="right", fontweight="bold")
    axF.plot([0.0, 1.0], [y - 0.052, y - 0.052], lw=0.5,
             color=COLORS["light_gray"], clip_on=False)
axF.set_xlim(0, 1)
axF.set_ylim(-0.03, 1.02)
axF.set_title("F  Run metrics at the Phase-20a refresh", fontsize=10, loc="left", pad=6)


In [ ]:
fig.savefig(OUT, dpi=300, bbox_inches="tight", facecolor="white")
print("wrote", OUT)
